# exp121_tabicl_artifact_diversity_audit train

CPU-only audit for TabICL and saved artifact-stack prediction diversity. This notebook does not train a model, rerun TabICL, or create a submission candidate.

## Contents

1. Setup and configuration
2. Input contract and source plan
3. Run diversity audit
4. Metrics and generated artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', config['experiment']['route'])
print('Status:', config['experiment']['status'])
print('GPU enabled:', config['runtime']['kaggle']['enable_gpu'])
print('Internet enabled:', config['runtime']['kaggle']['enable_internet'])
print('Sample submission:', paths.sample_submission_path)
print('Artifacts:', paths.artifacts_dir)


## 2. Input contract and source plan


In [ ]:
sample = pd.read_csv(paths.sample_submission_path)
print('Sample rows:', len(sample))
print('Sample columns:', list(sample.columns))
display(sample.head())

source_plan = pd.DataFrame(config['audit']['source_roots'])
anchor_plan = pd.DataFrame(config['audit']['explicit_submissions'])
display(source_plan[['name', 'role', 'family', 'paths']])
display(anchor_plan[['name', 'role', 'family', 'path']])


## 3. Run diversity audit


In [ ]:
from tabicl_artifact_diversity_audit import run_audit

metrics = run_audit()
print(json.dumps(metrics, indent=2, sort_keys=True))


## 4. Metrics and generated artifacts


In [ ]:
artifact_paths = {name: Path(path) for name, path in metrics['artifacts'].items()}
for name, path in artifact_paths.items():
    print(name, path, 'exists=', path.exists())

inventory = pd.read_csv(artifact_paths['inventory'])
pairwise = pd.read_csv(artifact_paths['pairwise'])
by_well = pd.read_csv(artifact_paths['by_well_distance'])

print('Inventory statuses:')
display(inventory.groupby(['record_type', 'status'], dropna=False).size().reset_index(name='count'))

if len(pairwise):
    display(pairwise.sort_values('rmse').head(20))
else:
    print('No pairwise rows. Mount at least one candidate/reference and one anchor to compute distances.')

if len(by_well):
    display(by_well.sort_values('rmse', ascending=False).head(20))
